# **EXPLANATION OF SCRIPTS**

(mainly from exercises)

<div class="alert alert-block alert-info">

### **Week 1**

Introdction to PyHPC
</div>

##### **Connecting and transferring files**

(exercise 1 of week 1)

This exercise established the basic workflow used every week. The key commands to remember:

```bash
scp helloworld.py s123456@login.hpc.dtu.dk:path/to/dir/   # Upload
scp s123456@login.hpc.dtu.dk:path/to/file.txt ./           # Download
ssh s123456@login.hpc.dtu.dk                               # Connect
linuxsh                                                     # Move to work node
```

Always run `linuxsh` after SSH login. The login node is shared among everyone connecting to the cluster so running computations there affects other users. `linuxsh` moves us to an interactive work node where it is safe to run code and short tests.

To initialise the course environment every session:

```bash
source /dtu/projects/02613_2025/conda/conda_init.sh
conda activate 02613

##### **Job scripts**

```bash
#!/bin/bash
#BSUB -J sleeper              # Job name (shown in bstat/bjobs)
#BSUB -q hpc                  # Queue to submit to
#BSUB -W 2                    # Wall-clock time limit in minutes
#BSUB -R "rusage[mem=512MB]"  # Memory per core
#BSUB -n 4                    # Number of cores
#BSUB -R "span[hosts=1]"      # All cores on the same node
#BSUB -o sleeper_%J.out       # stdout file (%J = job ID)
#BSUB -e sleeper_%J.err       # stderr file

sleep 60
```

Submit with `bsub < submit.sh`. The `#BSUB` lines are directives to the LSF scheduler, they look like comments to bash but are read by `bsub`.

Memory gotcha `rusage[mem=512MB]` is memory per core. With `-n 4`, the actual reserved memory is $4 \times 512  \: MB = 2 \: GB$. This is easy to forget and can cause jobs to be killed if we under-request.


##### **Monitering jobs**

```bash
bstat          # Compact overview: job ID, status, elapsed time
bjobs          # More detail: includes TIME_LEFT (wall time remaining)
bjobs -p       # Shows pending reason - use this when a job won't start
bkill JOBID    # Kill a job
```

Job states to know are `PEND` (waiting for resources), `RUN` (executing), `DONE` (finished successfully). `bstat` does not show the wall time limit, `bjobs` does via the `TIME_LEFT` column.

If a job exceeds its wall time, LSF kills it and the output file will contain `TERM_RUNLIMIT: job killed after reaching LSF run time limit` instead of `Successfully completed.`

##### **Selecting a specific CPU model**

```bash
#BSUB -R "select[model==XeonGold6226R]"
```

This is essential for reproducible timing results. Without pinning the CPU model, the same job could run on hardware with different clock speeds and cache sizes, making benchmark comparisons meaningless. Every timing exercise from week 3 onwards uses this. So after the job runs, `lscpu` in the script confirms the actual CPU used.

Find available CPU types with:
```bash
nodestat -f hpc
```

Note the slight naming inconsistency in `nodestat` output the type may appear as `XeonGold6226R` with a dash, while `select[model==...]` uses an underscore in some cases - worth checking with `nodestat` first.

##### **Core count limits**


Requesting too many cores on a single node (`span[hosts=1]`) will cause the job to pend indefinitely. The error from `bjobs -p` is `Not enough job slot(s)`. The `hpc` queue has nodes with up to around 32 cores, so 64 cores on a single host will never start. This is a common mistake in later parallelism exercises when scaling up process counts.

##### **Job summary email**

Adding `#BSUB -B` (start notification) and `#BSUB -N` (end notification) to a script sends email on job start and end. The end email includes a job summary with `Run time`, `Max Memory`, and `CPU time` which is useful for right-sizing resource requests. If `Run time` or `Max Memory` is far from what we requested, adjust the script before resubmitting repeatedly. The "Successfully completed." line is the key indicator that nothing went wrong.

<div class="alert alert-block alert-info">

### **Week 2**

Python Bootcamp
</div>


Week 2 exercises are primarily about getting comfortable with Python syntax and running scripts on the HPC. The content is foundational and directly tested in exam questions that ask you to read and reason about Python code. Key things the exercises cover:

**Command-line arguments (`sys.argv`):** every HPC script uses `sys.argv[1]` to receive its job index or input path from the shell. The pattern `int(sys.argv[1]) - 1` to convert from 1-based LSF indexing to 0-based Python is used in every job array exercise.

**List comprehensions:** faster and more Pythonic than explicit `for` loops with `append`. Use them whenever you are building a list from a transformation.

```python
# Prefer this:
squares = [x*x for x in data]

# Over this:
squares = []
for x in data:
    squares.append(x*x)
```

**Generators and `yield`:** the key advantage is that they do not materialise the entire sequence in memory. The course uses generators extensively for processing large files (week 7). The pattern is:

```python
def my_generator(data):
    for item in data:
        yield process(item)   # one value at a time

# Consume lazily:
for result in my_generator(big_data):
    ...

# Materialise if needed:
all_results = list(my_generator(big_data))
```

**Lambda functions:** anonymous single-expression functions. Used most often as the `key=` argument to `sorted` or `list.sort`:

```python
# Sort pairs by second element:
pairs = [(1, 'one'), (2, 'two'), (3, 'three')]
pairs.sort(key=lambda pair: pair[1])   # alphabetical by name

# Sort by absolute value:
nums = [-3, 1, -2, 4]
sorted(nums, key=lambda x: abs(x))   # [1, -2, -3, 4]
```

**Basic NumPy operations:** the exercises use NumPy throughout. The most important operations to know cold are: `np.zeros`, `np.ones`, `np.arange`, array arithmetic, `a.shape`, `a.dtype`, `a.nbytes`, slicing (`a[1:3]`, `a[:, 0]`), and `a.sum(axis=...)`. Views vs copies is tested indirectly in week 4 (section 4.9 in book notes).

**Data structures and O(1) lookup:** if you need to repeatedly check membership (`if x in collection`), use a `set` or `dict`, not a `list`. This pattern appears in exam questions about which code has a performance problem.

```python
# SLOW: O(N) per check, O(N^2) total for N items
valid_ids = [1, 2, 3, ...]  # list
for item in data:
    if item.id in valid_ids:   # O(N) scan each time!
        ...

# FAST: O(1) per check
valid_ids = {1, 2, 3, ...}   # set
for item in data:
    if item.id in valid_ids:   # O(1) hash lookup
        ...
```

<div class="alert alert-block alert-info">

### **Week 3**

The Memory Hierarchy
</div>


##### **Cache effects**

(exercise 1 of week 3)

##### **Row versus column access**

```python
mat = np.random.rand(SIZE, SIZE)

for _ in range(n_repeat):
    mat[0, :] * 1.01   # Row access

for _ in range(n_repeat):
    mat[:, 0] * 1.01   # Column access
```

Both operations double the same number of elements in a square matrix, so common sense says they should take the same time but they do not.

NumPy arrays are stored row-major (C-order) so elements in the same row are contiguous in memory. When we access `mat[0, :]`, all elements are adjacent so the CPU loads one cache line (typically 64 bytes = 8 float64 values) and uses all of them. When we access `mat[:, 0]`, each element is a full row-width apart in memory. Every access lands in a different cache line, most of which get evicted before they are reused. This is a classic case of poor spatial locality.

The effect is invisible at small sizes (for instance, $100 \times 100 = 80 \: KB$) because the whole matrix fits in L1/L2 cache regardless. At large sizes (so like $10,000 \times 10,000  \approx 800 \: MB$), the matrix far exceeds all cache levels, and column access incurs a full cache miss on every single element - a 16x slowdown compared to row access on the hardware used.

##### **Why measurements must be run as batch jobs**

The lecture explicitly warns: do not run cache experiments on your own laptop. Background processes, OS interrupts, and shared CPU usage all interfere with cache state in unpredictable ways. Submitting to a batch job on a dedicated node with a pinned CPU model (`select[model==XeonGold6126]`) ensures no other process is evicting your cache lines during measurement. This is also why 1000 repetitions are used which was to average out any remaining noise.


##### **Reading cache sizes from `lscpu`**

```
L1d cache:   32K
L2 cache:    1024K
L3 cache:    19712K
```

These numbers let you predict exactly where the performance cliff will appear on the MFLOP/s plot. The transition from "matrix fits in L1" to "matrix spills into L2" is where row and column access start to diverge. The transitions at L2 to L3 and L3 to RAM cause further stair-step drops. The plot is really a physical map of the memory hierarchy made visible through timing data.

The conversion from array size to kilobytes is a $SIZE \times SIZE$ float64 matrix occupies `SIZE^2 x 8` bytes. A single row of length SIZE occupies `SIZE x 8` bytes, which helps explain why the row vector experiment (part 4) uses a 1xSIZE array and can probe much larger sizes before hitting cache limits.


##### **The MFLOP/s metric**

To compare fairly across different array sizes, raw time is converted to MFLOP/s (millions of floating point operations per second):

```
MFLOP/s = (number of operations) / (time in seconds) / 1e6
```

For doubling a vector of length $N$, there are $N$ multiplications, so `FLOP/s = N / t`. This normalises the metric so that a drop in MFLOP/s directly means the hardware is spending more time waiting for data rather than computing so the cache miss penalty is made visible.

##### **The row vector experiment**

```python
mat = np.random.rand(1, SIZE)
for _ in range(n_repeat):
    mat[0, :] * 2
```

Replacing the 2D matrix with a 1D row vector isolates the effect of the memory hierarchy on a single sequential access pattern. The performance shows a clear staircase where flat and fast while the vector fits in L1, a drop at L1 to L2 boundary ($32 \: KB$), another at L2 to L3 ($1 \: MB$), and another at L3 to RAM ($20 \: MB$). For very small arrays, Python's own overhead (function call, loop body) dominates the measured time, which is why performance initially appears to *increase* as the array grows - the actual computation starts to outweigh the constant Python overhead.

##### **Efficient data storage with Blosc**

(exercise 2 of week 3)

##### **Three array types**

The exercise uses three types of 3D arrays of size $N\times N\times N$ with `dtype='uint8'`:

```python
zero_arr    = np.zeros((n, n, n), dtype='uint8')
tiled_arr   = np.tile(np.arange(256, dtype='uint8'), ...).reshape(n, n, n)
random_arr  = np.random.randint(0, 256, size=(n,)*3, dtype='uint8')
```

These three types are chosen deliberately to represent the full spectrum of compressibility. Zeros are trivially compressed (run-length encoding "$N$ zeros"). Tiled values have a regular repeating pattern that compressors exploit extremely well. Random values have maximum entropy so no pattern to exploit and are essentially incompressible.

##### **The `os.sync() call**


```python
def write_blosc(arr, file_name, cname="lz4"):
    b_arr = blosc.pack_array(arr, cname=cname)
    with open(f"{file_name}.bl", "wb") as w:
        w.write(b_arr)
    os.sync()   # Force OS to flush write buffers to disk
```

This is essential for fair benchmarking. Without it, the OS may keep written data in an in-memory write buffer, making the write appear instantaneous when it has not actually hit the disk. `os.sync()` forces all pending writes through to storage before timing ends. The same problem exists for reads, as after writing, the file may still be in OS read buffers, making a subsequent `read` far faster than a true cold read from disk. This is why the exercises specify running on a fresh batch job so the OS buffers are cold at job start.


##### **When Blosc wins and when it does not**

The key insight from chapter 6.2, confirmed by the exercise results, is:

| Array type | NumPy write | Blosc write | Blosc win? |
|---|---|---|---|
| zeros | $7.5 \: s$ | $0.5 \: s$ | Yes (15x) |
| tiled | $7.5 \: s$ | $0.5 \: s$ | Yes (15x) |
| random | $7.5 \: s$ | $8.1 \: s$ | No (slightly slower) |

For zero and tiled arrays, Blosc compresses the data so aggressively that far less data is written to disk, and the time saved on disk I/O exceeds the time spent compressing in the CPU. This is the key principle from the lecture: *"Speed up if time to compress is less than time to write extra data."* For random data, compression fails to reduce the file size, so we pay the compression overhead with no I/O benefit hence NumPy wins.

Disk space differences are even more dramatic as the tiled array is 200x smaller with Blosc, the zero array 250x smaller, and the random array is essentially the same size. This means Blosc is almost always worth using for real-world scientific data, which tends to have patterns.

##### **Lz4 versus Zstandard**

| Algorithm | Compression time | Compressed size |
|---|---|---|
| LZ4 | $527 \: ms$ | $5,204 \: KB$ |
| Zstandard (zstd) | $919 \: ms$ | $366 \: KB$ |

LZ4 is designed for speed as it compresses and decompresses extremely fast at the cost of compression ratio. Zstandard achieves a 14x better compression ratio than LZ4 but takes 1.75x longer to compress. The choice depends on the bottleneck, if disk I/O time dominates (slow HDD), then the smaller file from `zstd` is worth the extra CPU time. If disk I/O is fast (NVMe SSD) or we need to compress frequently, `lz4` is the better trade-off. This is the same cost-benefit reasoning applied throughout the course: *measure your specific bottleneck before choosing a strategy*.

<div class="alert alert-block alert-info">

### **Week 4**

Profiling and High-Performance NumPy
</div>


##### **Broadcasting**

(exercise 1 of week 4)

Broadcasting is NumPy's mechanism for performing element-wise operations on arrays with different shapes, without explicitly copying data. The rules are that NumPy compares dimensions from the right, and dimensions are compatible if they are equal or one of them is 1. Here, a size-1 dimension is "stretched" to match the other but conceptually, not in memory.

The three small exercises in this week 4 build up the same core skill which is adding a dimension with `None` or `np.newaxis` to turn a 1D vector into a 2D row or column vector, enabling pairwise operations across all combinations.

`standardize_rows` subtracts a mean vector from every row and divides by std. Since `data` has shape `(N, M)` and `mean`/`std` have shape `(M,)`, NumPy broadcasts the 1D arrays across all $N$ rows automatically so no loop needed.

`outer` computes the outer product of two vectors of length $N$ and $M$. The trick is:

```python
x[:, None] * y[None, :]
# Shape: (N, 1) * (1, M) → (N, M)
```

`x[:, None]` gives shape `(N, 1)` so a column vector. `y[None, :]` gives shape `(1, M)` so a row vector. Broadcasting multiplies every element of $x$ against every element of $y$, producing the full $N \times M$ outer product matrix with no Python loop.

`distmat_1d` generalises the same idea to absolute differences:

```python
np.abs(x[:, None] - y[None, :])
# Shape: (N, 1) - (1, M) → (N, M)
```

This is the foundational pattern used repeatedly later in the Haversine exercise. 


##### **High performance Haversine**

(exercise 2 of week 4)

##### **The profiling workflow (two levels)**

The exercise deliberately uses two profilers in sequence, which is the recommended workflow from chapter 2:

```bash
# Step 1: Function-level profiling - find which functions are slow
python -m cProfile -s cumulative script.py

# Step 2: Line-level profiling - find which lines inside a slow function are slow
kernprof -l script.py
python -m line_profiler -rmt "script.py.lprof"
```

`cProfile` has very low overhead and tells us *which function* is the bottleneck. `line_profiler` is much slower (it instruments every line) but tells us *exactly which line inside that function* costs the most. As chapter explains, always start with `cProfile` to narrow the search space, then use `line_profiler` only on the confirmed bottleneck.

The `@profile` decorator needed by `kernprof` is only recognised when running under `kernprof` as it does not exist in normal Python execution. This means that we add it just for profiling runs, or guard it with a try/except.


##### **One loop for broadcasting (80x speedup)**


The original two-loop version:

```python
for i in range(len(p1)):
    for j in range(len(p2)):
        dsin2 = np.sin(0.5 * (p1[i] - p2[j])) ** 2
        ...
        D[i, j] = 2 * np.arctan2(...)
```

This one calls NumPy functions inside a Python loop so the worst of both worlds. Each call to `np.sin`, `np.cos`, `np.arctan2` has Python function-call overhead, and there are $N^2$ such calls. The one-loop version:

```python
for i in range(len(p1)):
    dsin2 = np.sin(0.5 * (p1[i] - p2)) ** 2    # operates on full row at once
    cosprod = np.cos(p1[i, 0]) * np.cos(p2[:, 0])
    a = dsin2[:, 0] + cosprod * dsin2[:, 1]
    D[i, :] = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
```

Reduces the number of NumPy calls from $O(N^2)$ to $O(N)$. Each call now operates on an entire row of length $N$ in C-level code. This gives the 80x speedup so the Python loop overhead is $N$ times less, and each NumPy operation processes $N$ elements at C speed.


##### **Line profiler output (what to focus on)**

So after the one-loop optimisation, the line profiler output shows:

```
Line #    % Time   Line Contents
  27       52.4%   dsin2 = np.sin(0.5 * (p1[i] - p2)) ** 2
  28       21.0%   cosprod = np.cos(p1[i, 0]) * np.cos(p2[:, 0])
  30       14.4%   row = np.arctan2(np.sqrt(a), np.sqrt(1 - a))
```

Lines 27 and 28 together account for $73 \%$ of runtime. This directs the next two optimisations precisely:

Optimisation 1 pre-computes `np.cos(p2[:, 0])` outside the loop. Inside the loop, `p2` never changes so computing its cosine $N$ times is pure wasted work. Pre-computing it once is both algorithmically correct and turns $N$ NumPy calls into 1.

```python
cos_p2 = np.cos(p2[:, 0])    # Compute once, outside loop
for i in range(len(p1)):
    cosprod = np.cos(p1[i, 0]) * cos_p2   # Now just a scalar × vector
    ...
```

Optimisation 2 replaces `arctan2(sqrt(a), sqrt(1-a))` with `arcsin(sqrt(a))`. These are mathematically identical via the trigonometric identity `arctan2(sqrt(a), sqrt(1-a)) = arcsin(sqrt(a))` for `a ∈ [0, 1]`. The `arcsin` version computes only one square root instead of two, cutting the cost of line 30 roughly in half. Together these save $25\%$ of total runtime so modest individually, but found precisely because line profiling told us exactly where to look.

##### **No-loop versus one-loop (memory bandwidth crossover)**

The fully vectorised no-loop version:

```python
dsin2 = np.sin(0.5 * (p1[:, None, :] - p2[None, :, :])) ** 2
cosprod = np.cos(p1[:, None, 0]) * np.cos(p2[None, :, 0])
D = 2 * np.arcsin(np.sqrt(dsin2[:,:,0] + cosprod * dsin2[:,:,1]))
```

Creates intermediate arrays of shape `(N, N, 2)`. For $N=5000$, that is $5000 \times 5000 \times 2 \times 8$ bytes = $400 \: MB$ of intermediate data. This must be allocated and written to RAM.

For small $N$ (distance matrix fits in L2 cache), the no-loop version is faster because there is no Python loop overhead at all. For large $N$, the one-loop version wins because it only ever materialises one row at a time, fitting easily in cache, whereas the no-loop version thrashes RAM with $400 \: MB$ of temporaries.

The crossover happens around the L2 cache boundary ($1 \: MB$), which corresponds to a distance matrix of roughly $350 \times 350$ float64 values. This is the key finding: *"more vectorised" does not always mean faster*. Whether it is faster depends on whether the intermediate arrays fit in cache, a hardware property, not a code property. So as the professor's note states: *you cannot always predict what code will be fastest - you have to measure.*

<div class="alert alert-block alert-info">

### **Week 5**

Parallelism Part 1
</div>


##### **Amdahl's law**

(exercise 1 of week 5 and 6)

$$S(p) = \frac{1}{(1-F) + \frac{F}{p}} = \frac{1}{B + \frac{1-B}{p}}$$

Where $F$ is the parallel fraction, $B = 1 - F$ is the serial fraction, and $p$ is the number of processors. The maximum theoretical speedup as $p \to \infty$ is simply:

$$S_\infty = \frac{1}{B} = \frac{1}{1 - F}$$

For the exercise the task has $20 \: s$ serial (file I/O) and $100 \: s$ parallelisable, so the parallel fraction is $F = 100/120 \approx 0.833$, and the maximum speedup is 6x. No matter how many processors we add, the 20-second serial part acts as an absolute floor for the runtime.


##### **Opimizing the bottleneck**

The exercise asks which is better: cutting the serial part from $20 \: s$ to $5 \: s$, or doubling the processors from 4 to 8.

With 4 processors and the serial part optimised to $5 \: s$, the new parallel fraction is $F = 100/105 \approx 0.952$, giving runtime $= 5 + 100/4 = 30 \: s$.

With 8 processors and the original serial part ($20 \: s$), runtime $= 20 + 100/8 = 32.5 \: s$.

Optimising the serial part wins - by a small margin here, but the principle scales dramatically. The Amdahl's law plot makes this vivid as with a parallel fraction of $50\%$, throwing infinite processors at a program gives at best 2x speedup. With $95\%$ parallel fraction, we can achieve 20x on real hardware. The serial bottleneck, not the number of processors, sets the ceiling. This is directly relevant to the Mandelbrot exercise where reducing the serial overhead (through chunking) improves the parallel fraction from $0.8$ to $0.98$.

##### **Parallel pi approximation**

(exercise 2 of week 5)

##### **Hierarchy overhead of three implementations**

Implementation 1 (serial) is baseline. No parallelisation overhead.

Implementation 2 (fully parallel - one task per sample):

```python
results_async = [pool.apply_async(sample) for i in range(samples)]
hits = sum(r.get() for r in results_async)
```

This submits 1,000,000 separate tasks to the pool - one per random sample. Each task does a trivial amount of work (one random point), but `apply_async` has a fixed overhead per submission: pickling arguments, sending them over an inter-process pipe, and returning the result the same way. With 1M tasks, this overhead completely dominates, and Implementation 2 is almost certainly *slower* than the serial version. The `time` command makes this visible: the `real` time exceeds `user` time on a single core.

Implementation 3 (chunked parallel):

```python
results_async = [pool.apply_async(sample_multiple, (chunk_size,))
                 for i in range(n_proc)]
```

This submits only `n_proc` tasks, each doing chunk_size = 100,000 samples. Now each process does substantial work and the inter-process communication is negligible. This is the correct version and will be fast. The lesson, directly from the lecture, is: *"Parallelization overhead is larger than runtime with the solution being chunking."*


##### **The `time` command output**

```bash
time python pi.py
real    0m5.2s
user    0m18.4s
sys     0m0.3s
```

`real` is wall-clock time. `user` is total CPU time across all cores. For the chunked parallel version the `user` > `real` means multiple cores are running simultaneously (good). For Implementation 2 the `real` $\approx$ `user` even with multiple processes means almost no real parallelism - the overhead of 1M IPC calls serialises execution. This is the smoking gun that tells us Implementation 2 is broken despite being "parallel". `sys` here is negligible, which is expected as we are not doing much I/O or system calls. Normally in this output `sys` is for kernel time, which includes things like file I/O, context switching, and inter-process communication. In a well-optimised parallel program, we want `sys` to be low compared to `user`, indicating that most of the time is spent doing actual computation rather than overhead.

##### **`apply_async` versus `pool.map`**

```python
# apply_async - manual, flexible
results = [pool.apply_async(f, (arg,)) for arg in data]
results = [r.get() for r in results]

# pool.map - convenient, automatic chunking parameter
results = pool.map(f, data, chunksize=N)
```

`pool.map` with a `chunksize` argument internally batches submissions, equivalent to the manual chunked approach. It is simpler to write and read. The trade-off is that `pool.map` is eager (blocks until all results are ready) and slightly less flexible than manual `apply_async` when tasks have variable completion times or when we need to process results as they arrive. For simple embarrassingly parallel problems, `pool.map` with an appropriate `chunksize` is the idiomatic choice.

##### **Fitting Amdahl's law to a speedup plot**

The exercise uses the measured speedup curve and fits $F$ by eye until the Amdahl curve matches. A fitted parallel fraction of $F = 0.945$ (serial fraction $B = 0.055$) gives a theoretical maximum speedup of $1/0.055 ≈ 18.2$. Real speedup will plateau below this due to additional overheads not captured by Amdahl's model (process startup, memory bandwidth limits, OS scheduling).

##### **Mandelbrot set**

(exercise 3 of week 5)

##### **Why even distribution is a bad strategy**

```python
# Bad: divide 800×800 points into n_proc equal chunks upfront
chunk = len(points) // n_proc
chunks = [points[i*chunk:(i+1)*chunk] for i in range(n_proc)]
```

The escape time varies enormously across the complex plane. Points well inside the Mandelbrot set require all 100 iterations. Points far outside escape on iteration 1 or 2. When we assign equal numbers of points to each process, some processes get unlucky and receive many slow interior points, while others finish quickly and sit idle waiting. This is load imbalance, and it directly limits speedup regardless of how many cores we have.


##### **Fixing with many small chunks (dynamic scheduling)**

```python
# Good: create many more chunks than workers
chunk_size = 100   # much smaller than points/n_proc
```

With more chunks than workers, a process that finishes its chunk quickly picks up the next available chunk from the pool queue. Slow chunks are spread across multiple processes rather than bottlenecking one. The pool acts as a task queue, and the workers self-schedule based on completion. This is sometimes called dynamic work distribution or work stealing.

The improvement in the speedup plot is dramatic so the parallel fraction rises from $0.8$ (fixed chunks) to $0.98$ (small chunks), pushing the theoretical maximum speedup from 5x to 50x. This perfectly demonstrates Amdahl's law in reverse but not the formula itself, but the idea that reducing the effective serial bottleneck (in this case, the idle time caused by load imbalance) is more impactful than adding more processors.

The general principle applies to any parallel workload with variable task duration: always chunk smaller than you think necessary, and let the pool scheduler handle load balancing automatically.

<div class="alert alert-block alert-info">

### **Week 6**

Parallelism Part 2
</div>


##### **Parallel tree reduction of "mean faces**

(exercise 1 of week 6)


The mean face computation is fundamentally a sum over 100,000+ images. Sum is a reduction operator as it is both commutative ($a + b = b + a$) and associative ($(a + b) + c = a + (b + c)$). As chapter 14 explains, these two properties are the exact requirements that make an operator parallelisable, because we can reorder and regroup the operands freely, we can split the array into independent subtasks and compute them simultaneously without affecting the final result. Here, a non-example is the power function `a ^ b ^ c`, which is not commutative and gives different results depending on evaluation order.

##### **The tree reduction pattern**

Rather than summing all images sequentially ($N−1$ additions, all serial), the tree reduction does it in rounds:

```
Round 1: arr[0] += arr[1], arr[2] += arr[3], arr[4] += arr[5], ...
Round 2: arr[0] += arr[2], arr[4] += arr[6], ...
Round 3: arr[0] += arr[4], ...
```

This takes $\log_2(N)$ rounds instead of $N−1$. With $N = 100,000$ images, that is 17 rounds instead of 99,999 sequential additions. Each round can be fully parallelised across all available cores since every pair-wise addition is independent.

The template code captures one step with the strided slice:

```python
def reduce_step(args):
    b, e, s, elemshape = args
    arr = tonumpyarray(shared_arr).reshape((-1,) + elemshape)
    arr[b:e:s] += arr[b+s//2:e:s]  # each element adds its neighbour
```

`b`, `e`, `s` define the start, end, and stride for a chunk of this round. The full reduction calls `pool.map` once per round, passing a new set of `(b, e, s)` arguments each time as the active range halves.


##### **Why shared memory is essential**

Normal `multiprocessing` sends data between processes by pickling and copying over an IPC pipe. For 100,000 images of shape $218\times 178\times 3$ at float32, the array is $10 \: GB$. Copying this between processes on every round would be prohibitively expensive and defeat the purpose of parallelism entirely.

`mp.RawArray` allocates a block of memory that all processes can access directly - no copying, no pickling:

```python
shared_arr = mp.RawArray(ctypes.c_float, data.size)
arr = tonumpyarray(shared_arr).reshape(data.shape)
np.copyto(arr, data)
```

The `tonumpyarray` helper wraps the raw memory as a NumPy array view using `np.frombuffer`. Worker processes access the same physical memory via the `init` function, which sets a global reference to it. Writes by one process are immediately visible to all others so no inter-process communication at all during computation, just a `pool.map` call per reduction round to coordinate which indices each worker handles.

##### **Slowdown beyond 24 cores (NUMA)**

The speedup plot will show a peculiar result that performance improves up to 24 cores but then *decreases* when using more. This is Non-Uniform Memory Access (NUMA), and it is the new concept introduced in week 6 beyond the basic parallelism from Week 5.

A server-grade CPU node (like the Xeon Gold 6342 used in the exercise) is physically two separate CPU chips (sockets) on one motherboard, each with its own local memory bank. The first 24 cores live on socket 0, the next 24 live on socket 1. When only socket 0's cores are running, all memory reads/writes go to socket 0's local memory. When socket 1's cores are also running, they must access the shared array which is allocated in socket 0's memory bank, travelling over a slower inter-socket interconnect. This makes memory accesses for those cores significantly slower, explaining the performance drop.


##### **Fixing with `numactlt --interleave=all`**

```bash
numactl --interleave=all python reduction.py <path>
```

`--interleave=all` tells the OS to distribute memory allocations in a round-robin fashion across both NUMA memory banks. Instead of the entire shared array sitting in one socket's memory, pages are spread across both banks. Now:

> Socket 0's cores access roughly half the pages locally and half remotely.

> Socket 1's cores access roughly half locally and half remotely.

Both sockets get balanced, symmetric access. No single socket is a memory bottleneck. The speedup curve then continues to grow properly beyond 24 cores.

It is worth noting from the lecture that `--interleave=all` can solve *scaling* (the curve no longer drops) but may not improve the single-core speed, since average memory latency stays the same or may even increase slightly for small datasets. The benefit only materialises when using many cores across both sockets.

##### **Chunk size trade-off**

The `chunk` variable in the template controls how many images each worker handles per `pool.map` call within a reduction step. Too small then high scheduling overhead (many tiny `apply_async` calls). Too large then fewer parallel tasks than cores, leaving cores idle. The professor found `chunk=64` to be a sweet spot for the 100K dataset, large enough to keep processes busy, small enough to keep all cores utilised throughout.


##### **Comparing against `np.sum`**

There is a final question that asks whether the parallel reduction beats `np.sum(data, axis=0)`. The result is faster, but not dramatically - roughly 2 to 4 times at most. This is because NumPy's `sum` is already highly optimised C code that exploits SIMD (vector) instructions internally, while the Python multiprocessing overhead and the cost of coordinating $log_2(N)$ rounds of `pool.map` adds its own overhead. The exercise demonstrates that writing parallel Python code that substantially beats a well-optimised single-threaded C implementation is genuinely difficult, so raw parallelism is not a free performance multiplier.

<div class="alert alert-block alert-info">

### **Week 7**

High-Performance Pandas and Apache Arrow
</div>


##### **Storage and reading files with Pandas**

(exercise 1 of week 7)

##### **Reading compressed versus uncompressed CSV files**

The first observation is that reading the zip directly with `pd.read_csv` ($11.1 \: s$) is slightly faster than unzipping first and then reading ($4.7 \: s + 12.7 \: s = 17.4 \: s$ total). Reading directly from the zip avoids the intermediate step and also saves disk space since no unzipped copy is created. Pandas is smart enough to infer the compression type from the file extension and handle it transparently.


##### **Measuring dataframe memory**

```python
df.memory_usage(deep=True).sum()
```

The `deep=True` argument is critical, because without it, pandas only reports the shallow size of each column's index structure and misses the actual memory consumed by object-typed columns (Python strings). The $2 \: GB$ result for a $120 \: MB$ zip file is a good illustration of why this matters as CSV text is compact, but when inflated into Python objects in RAM, memory balloons dramatically.


##### **Five techniques for memory reduction**

The `summarize_columns` function reveals that all the memory cost sits in just a few columns. The reduction strategy column-by-column:

`created` and `observed` (string to datetime): these are the biggest wins. each goes from $600 \: MB$ (object) to $62 \: MB$ (datetime64). The reason object columns are so expensive is that each value is a separate Python string object with its own heap allocation and reference count overhead. Here, a datetime64 array is a contiguous block of 8-byte integers so dramatically more compact.

`parameterId` (string to category): only 47 unique values out of 8 million rows. As a category dtype, pandas stores those 47 strings once in a dictionary and the column becomes an array of small integer indices pointing into it. Furthermore, as chapter 7.1 explains, a column with a small number of possible values compared to its length is the ideal candidate for categorical encoding.

`stationId` (int64 to int16): here, only 247 unique station IDs, but the values range up to $34339$ - this is too large for int8 (max 127), but fits in int16 (maximum is $32767$). This halves the column from 64-bit to 16-bit integers.

`value` (float64 to float32): the test `(df["value"] - df["value"].astype('float16')).abs().max()` returns `inf`, meaning float16 loses too much precision for some values (overflow). float32 gives a maximum error of around $6e-5$, which is acceptable for precipitation measurements. This halves the column from 8 bytes to 4 bytes per value.

`coordsx` / `coordsy` (float64 to category): coordinates represent physical station locations, there are only $224/219$ unique values across 8 million rows. Encoding as category means the actual floats are stored once in a lookup table, the column stores only the index. This is a non-obvious but effective application of categorical encoding to numeric data.

The key lesson from chapter 7.1 is that pandas' default behaviour on `read_csv` is to use the broadest safe type for each column, which is almost always wasteful for real datasets. Profiling with `summarize_columns` before committing to a schema is the standard approach.

##### **Reading files with Arrow**

(exercise 2 of week 7)

##### **PyArrow CSV Loading**


```python
from pyarrow import csv
table = csv.read_csv('2023_01.csv')
```

PyArrow reads the same CSV 3.7x faster than pandas ($3 \: s$ versus $11.1 \: s$). PyArrow's CSV reader is implemented in C++ and uses multiple threads internally, whereas pandas' reader is primarily single-threaded Python. Converting back to pandas adds $700 \: ms$, but the total ($3.7 \: s$) is still well under plain pandas ($11.1 \: s$).


##### **Why PyArrow table is smaller than Pandas dataframe in memory**

The PyArrow table ($507 \: MB$) is smaller than the converted pandas DataFrame ($919 \: MB$) even though they hold the same data. The key difference is how strings are stored. In PyArrow, all strings in a column are laid out contiguously in a single byte buffer, with a separate offset array pointing to where each string starts and ends. In pandas (before version 3.x), string columns are arrays of Python object pointers, so each string is a separate heap-allocated Python object with full Python overhead (reference count, type pointer, and so on). This is why `parameterId` alone accounts for hundreds of MB in pandas but far less in Arrow.

Note that Pandas 3.x (used in the course) uses a `str` dtype for text columns which is more memory-efficient than the old `object` dtype, but converting to `datetime` and `category` is still beneficial.

##### **Applying memory reductions on load with PyArrow**

```python
convert_options = csv.ConvertOptions(
    column_types={
        'value': pa.float32(),
        'parameterId': pa.dictionary(pa.int32(), pa.string()),
        'coordsx': pa.dictionary(pa.int32(), pa.float64()),
        'coordsy': pa.dictionary(pa.int32(), pa.float64()),
    }
)
table = csv.read_csv(fname, convert_options=convert_options)
```

`pa.dictionary(pa.int32(), pa.string())` is Arrow's equivalent of pandas' category dtype, an index array pointing into a value dictionary. Applying these conversions on load reduces the Arrow table to $312 \: MB$ and means the data never exists in memory in its large form. This is a general principle as it is always cheaper to avoid allocating memory than to allocate and then free it.


##### **Parquet files**

(exercise 3 of week 7)

##### **From CSV to Parquet**

The resulting Parquet file is $86 \: MB$, compared to the $120 \: MB$ zip or $700 \: MB$ unzipped CSV. Parquet achieves this through two mechanisms: binary encoding (numbers as raw bytes rather than ASCII digits) and column-level compression (each column is compressed independently with codecs like Snappy or Zstd, which work well on homogeneous typed data).

The performance difference on read is substantial:

| Format | Read time |
|---|---|
| CSV (pandas) | $ \sim 11 \: s$ |
| Parquet (pandas `read_parquet`) | $ \sim 1 \: s$ |
| Parquet (PyArrow `read_table`) | $ \sim 0.4 \: s$ |

The speedup comes from not having to parse text. Reading binary data with known types is purely a memory bandwidth problem. Just as chapter 8.2 explains, Parquet is also columnar, if we only need two columns, we only read those two columns' bytes from disk, whereas CSV always requires reading every byte of every row.


##### **Fast operations in Pandas**

(exercise 4 of week 7)

##### **The three implementation (performance hierarchy)**

This exercise worked through three approaches to the same computation, each representing a distinct performance tier in pandas. Raw Python (`iloc` row iteration) was $5.5 \: s$ for 100K rows.

```python
for i in range(len(df)):
    row = df.iloc[i]  # Python call per row
    if row['parameterId'] == 'precip_past10min':
        total += row['value']
```

Every call to `df.iloc[i]` is a Python function call that constructs a new pandas Series object. 100k rows = 100,000 object constructions. This is the worst of both worlds as Python loop overhead plus pandas overhead on every iteration. Chapter 7.2 explicitly warns against row iteration strategies like this.

`apply()` is 7x faster giving $780 \: ms$ for 100K:

```python
df.apply(lambda row: row['value'] if row['parameterId'] == 'precip_past10min' else 0.0, axis=1).sum()
```

`apply` is still iterating rows under the hood, but with less Python overhead per row than manual `iloc`. It's marginally faster but still not truly vectorised. For the full dataset it takes $46 \: s$ which is still very slow.

Now, vectorised boolean indexing is 21x faster than raw, $270 \: ms$ for 100K and $400 \: ms$ full dataset:**

```python
df[df['parameterId'] == 'precip_past10min']['value'].sum()
```

1. `df['parameterId'] == 'precip_past10min'` creates a boolean mask array in one C-level operation.
2. The boolean index selects the relevant rows without a Python loop.
3. `.sum()` runs as a C-level reduction.

No Python object is created per row. The entire operation runs in NumPy/C. This is the idiomatic pandas approach and the one chapter 7.2 recommends as the default.


##### **Index-based lookups**

```python
df_pid = df.set_index('parameterId').sort_index()
precip = df_pid.loc['precip_past10min']['value'].sum()
```

Excluding the time to build the index, this takes just $3 \: ms$  so 130x faster than vectorised boolean indexing, 1,500x faster than `apply`. With the index, pandas can jump directly to the relevant rows using a binary search ($O(\log n)$) rather than scanning the entire column ($O(n)$).

However, building the index takes $6.6 \: s$ so slower than just doing the vectorised scan once. The index is only worth building if we plan to run many queries for different `parameterId` values. This is the classic trade-off between lookup tables and linear scan as an index amortises its construction cost over repeated use, exactly as described in chapter 7.2.1.

<div class="alert alert-block alert-info">

### **Week 8**

Storing Big Data
</div>


##### **Datafram chunking**

(exercise 1 of week 8)

##### **Pandas chunking**

```python
df_chunks = pd.read_csv(infile, chunksize=C)
for chunk in df_chunks:
    total += chunk[chunk['parameterId'] == 'precip_past10min']['value'].sum()
```

1. Pass `chunksize=C` to `read_csv` so instead of a DataFrame, we get back a `TextFileReader` generator.
2. Iterate over it chunk by chunk, each `chunk` is a regular DataFrame with at most `C` rows.
3. Accumulate the partial results into `total`.

The key insight is that at any moment only one chunk (C rows) is in RAM, not the entire file. This is the whole point so the full CSV is around $2 \: GB$, but peak memory usage at `chunksize=10,000` is only $130 \: MB$. The program trades some code complexity for a drastically smaller memory footprint.

The chunk size results show a clear trade-off:

| Chunk size | Runtime | Peak memory |
|---|---|---|
| $1,000$ | $27.6 \: s$ | $131 \: MB$ |
| $10,000$ | $16.7 \: s$ | $137 \: MB$ |
| $100,000$ | $15.9 \: s$ | $197 \: MB$ |
| $1,000,000$ | $16.5 \: s$ | $559 \: MB$ |
| No chunking | $17.2 \: s$ | $2,040 \: MB$ |

We see at `chunksize=1,000`, overhead from processing thousands of tiny DataFrames dominates. Then above `10,000`, runtime plateaus so we are not losing performance compared to loading the whole file, but we are using 15x less memory. Chapter 8.3 makes this exact point that chunking is the standard technique for processing larger-than-memory tabular data without significant performance loss.


##### **CSV to chunked parquet conversion**

```python
writer = None
for chunk in df_chunks:
    chunk_table = pa.Table.from_pandas(chunk)
    if first:
        writer = pq.ParquetWriter(outfile, schema=chunk_table.schema)
        first = False
    writer.write_table(chunk_table)
writer.close()
```

1. Read the CSV in chunks via pandas.
2. Convert each chunk to a PyArrow Table.
3. Infer the schema from the first chunk only, necessary because `ParquetWriter` needs a fixed schema upfront.
4. Write each chunk as a separate row group inside the single Parquet file.
5. Close the writer to flush and finalise the file.

The schema is initialised from the first chunk rather than hardcoded because some column types (particularly nullable integers) can change between chunks when nulls are present. The resulting `.parquet` file has one row group per chunk, these are the units that can be read back individually.

##### **Reading parquet row groups**

```python
pf = pq.ParquetFile(fname)
for i in range(pf.num_row_groups):
    group = pf.read_row_group(i).to_pandas()
    total += precip(group)
```

Reading the chunked Parquet file cuts runtime from 16 seconds (Pandas CSV) to 5 seconds so a times 3 speedup. Adding column selection (`columns=['parameterId', 'value']`) brings it further to $1.2 \: s$, a 13 times speedup over the chunked CSV baseline.

The two reasons are:

> Parquet is binary and typed - no CSV parsing overhead, no type inference.

> Column pruning - Parquet is columnar, so `read_row_group(i, columns=[...])` physically reads only the relevant columns from disk. In a row-oriented format like CSV, we must read every column even if we throw most away. This is the core advantage of Parquet described in chapter 8.2 that data is organised by column, so we only pay for what we use.


##### **Mandelbrot memmap**

(exercise 2 of week 8)

##### **Creating and writing a memory-mapped array**

```python
mm = np.memmap('mandelbrot.raw', mode='w+', shape=(N, N), dtype='int32')
```

1. `mode='w+'` creates a new file (or overwrites an existing one) and opens it for both reading and writing.
2. `mode='r+'` opens an existing file for read/write. `mode='r'` opens read-only.
3. `mm` looks and behaves exactly like a normal NumPy array - you index and assign to it with standard syntax.

What actually happens under the hood is the OS maps the file into the virtual address space. When we read or write `mm[i, j]`, the OS transparently brings the relevant page from disk into RAM (a page fault), and dirty pages are flushed back to disk when evicted or when the program ends. So as the week 8 lecture diagram shows, there is a "fake" memory region that maps directly to the data file - the CPU thinks it is reading RAM, but the backing store is on disk. This is why the array can be larger than physical RAM.

Parallel writing works safely because each process writes to a different, non-overlapping slice of the array. Since each slice corresponds to different pages on disk, there are no write conflicts and no need for locks.


##### **Downsampling with memmap**


```python
mm = np.memmap('mandelbrot.raw', mode='r', shape=(N, N), dtype='int32')
downsampled = mm[::step, ::step]
```

1. Open the existing file in read-only mode.
2. Slice with `[::step, ::step]` to take every `step`-th row and column.

The crucial observation from the timing results is that only the accessed pages are loaded from disk, not the entire array. With `step=16`, we read 1/256th of the data, and both memory use and runtime drop proportionally. This is memory mapping's key advantage over `np.load` so again we pay only for what we touch. The full $10000 \times 10000$ array is around $400 \: MB$, but at `step=16` the program uses a fraction of that.


##### **Mandelbrot Zarr**

##### **Creating a Zarr array**

```python
import zarr
store = zarr.open('mandelbrot.zarr', mode='w',
                  shape=(N, N), dtype='int32',
                  chunks=(chunk_size, chunk_size))
```

1. Zarr creates a directory on disk, not a single file. Each chunk is a separate file inside it.
2. `chunks=(C, C)` defines how the 2D array is partitioned. A $1000 \times 1000$ array with $200 \times 200$ chunks has 25 chunk files.
3. The array is not filled yet - uninitialized chunks occupy almost no space (only a `.zarray` metadata file exists).

This directory-per-array design is what enables parallel writes, since chunks are separate files, multiple processes can each write to their own chunk file simultaneously. HDF5, the traditional alternative, uses a single file and does not support concurrent writes. As the lecture notes says: *"Can read and write to chunks in parallel! (write not possible in HDF5)"*.


##### **Chunk size trade-offs**

The chunk size experiment on the $1000 \times 1000$ Mandelbrot array reveals two competing forces:

| Chunk size | Runtime | Stored size |
|---|---|---|
| $10$ | $24.2 \: s$ | $285 \: MB$ |
| $25$ | $3.6 \: s$ | $38 \: MB$ |
| $50$ | $1.1 \: s$ | $9.5 \: MB$ |
| $100$ | $1.3 \: s$ | $2.4 \: MB$ |
| $200$ | $4.3 \: s$ | $656 \: KB$ |

Runtime has a sweet spot around 50 to 100. Too small leads to too many tiny chunks, parallelism overhead from spawning a process per chunk dominates. Too large = fewer chunks than available cores, parallelism is wasted (cannot use all 24+ processes on 25 total chunks).

Storage size decreases monotonically as chunks grow. This is because Zarr compresses each chunk independently using Blosc/LZ4. Larger chunks contain more of the uniform "outside the Mandelbrot set" region, which compresses extremely well. Tiny chunks waste space on per-chunk metadata and compress less effectively. The lecture slide captures this: *"Speed up if time to decompress/compress is less than time to read/write extra data!"*

The chunk shape should also match the access patterns,as the lecture shows, tall narrow chunks are good for column-wise access, short wide chunks for row-wise, and square chunks for 2D neighbourhood access. Since the Mandelbrot exercise accesses the array in square blocks, the square chunk shape is natural.

Compared to the memmap version ($5.3 \: MB$ raw and $0.8 \: s$), the Zarr format at chunk size 100 ($2.4 \: MB$ and $1.3 \: s$) uses less than half the disk space at only a small runtime cost which is a compelling trade-off as data sizes grow.

<div class="alert alert-block alert-info">

### **Week 9**

Numba and GPU computing

</div>


##### **CPU matrix multiplication**

(exercise 1 of week 9)

##### **`@jit(nopython=True)` - JIT Compiling the Loop**


```python
from numba import jit

@jit(nopython=True)
def matmul_jit(A, B):
    ...
```

1. Add the decorator - Numba compiles this to native machine code the first time it runs.
2. `nopython=True` forces Numba to compile *entirely* without falling back to Python - if it cannot, it fails loudly. This is the mode we want as it guarantees no Python interpreter overhead.
3. Run once before timing to trigger compilation, then time the compiled version.

The 470x speedup over pure Python comes from the fact that Python loops are extremely slow, as each iteration has interpreter overhead, type checking, and object lookups. Numba eliminates all of that and generates tight native code equivalent to what a C compiler would produce.


##### **Loop re-ordering for cache efficiency**

The original loop order is `i → j → k`. The innermost loop body is:

```python
C[i, j] += A[i, k] * B[k, j]
```

NumPy arrays are stored row-major (C-order): elements in the same row are contiguous in memory. So as `k` increments:

- `A[i, k]` steps along a row = cache friendly
- `B[k, j]` steps down a column = cache unfriendly - each access is a stride of the full row width apart in memory, causing repeated cache misses.

The optimised fix is to swap the `j` and `k` loops to `i → k → j`:

```python
for i in range(A.shape[0]):
    for k in range(A.shape[1]):
        for j in range(B.shape[1]):
            C[i, j] += A[i, k] * B[k, j]
```

Now the innermost loop steps along a row of both `B` and `C` so fully cache friendly. The around 6x speedup is purely from better cache utilisation - same number of FLOPs, same hardware, just better memory access patterns. This is a direct application of the Week 3 memory hierarchy content.

The performance plot shows clear dips at cache boundary sizes (L1 to L2 to L3 to RAM), exactly illustrating how performance degrades when working data no longer fits in each cache level.

##### **CUDA vector addition**

(exercise 2 of week 9)

##### **The kernel**

```python
@cuda.jit
def add_kernel(x, y, out):
    i = cuda.grid(1)
    out[i] = x[i] + y[i]
```

1. Decorate with `@cuda.jit` - this is a GPU kernel function, not a regular Python function.
2. Each thread computes exactly one element: `out[i] = x[i] + y[i]`.
3. `cuda.grid(1)` returns this thread's unique global index across the entire grid.

This is the canonical embarrassingly parallel pattern: each element is independent, so we can assign one thread per element.

```python
threadsperblock = 256
blockspergrid = (n + threadsperblock - 1) // threadsperblock
add_kernel[blockspergrid, threadsperblock](x, y, out)
```

The ceiling division ensures all `n` elements are covered even when `n` is not a multiple of 256.


##### **The three memory scenarios**


Exercises of week 9 is deliberately structured to isolate and measure the cost of data movement. The progression is:

Scenario 1 - Plain NumPy arrays (around $6 \: ms$): Numba automatically transfers the arrays to GPU memory before the kernel and back after. This happens on *every call* inside the timing loop, so we are measuring transfer + compute + transfer 200 times.

Scenario 2 - Pinned memory (around $3.5 \: ms$): `cuda.pinned` marks the CPU memory as non-swappable (page-locked). Normally the OS can swap memory pages out to disk during a transfer, which would interrupt it. Pinned memory allows the GPU's DMA engine to transfer data directly without involving the CPU, roughly halving transfer time. Still doing transfers on every call, but they're faster.

Scenario 3 - GPU-resident arrays (around $0.042 \: ms$): arrays are moved to the GPU *once before timing*. Inside the loop, there are zero transfers - just kernel computation. The 140x speedup over pinned memory reveals that around $98.8\%$ of total runtime in scenario 1 was memory transfer, and only around $1.2\%$ was actual computation.

So for work-light kernels (one addition per element), the PCIe transfer cost completely dominates. The moral directly maps to what chapter 9 states: *"The cost of transferring data to and from the GPU memory can have a massive effect on performance, especially if the amount of computation that we do on the GPU is limited."*


##### **CUDA matrix multiplication**

(exercise 3 of week 9)

##### **The 2D kernel**

```python
@cuda.jit
def matmul_kernel(A, B, C):
    i, j = cuda.grid(2)
    if i < C.shape[0] and j < C.shape[1]:
        tmp = float32(0.)
        for k in range(A.shape[1]):
            tmp += A[i, k] * B[k, j]
        C[i, j] = tmp
```

1. Use `cuda.grid(2)` to get a 2D position - thread `(i, j)` computes `C[i, j]`.
2. Bounds check (`if i < ... and j < ...`) guards against threads launched beyond the matrix dimensions (since `blocks × threads` is rounded up).
3. Accumulate into a local `tmp` variable rather than writing to `C[i, j]` every iteration, this avoids repeated slow global memory writes inside the loop. Only one write at the end.

The 2D grid is launched like:

```python
threadsperblock = (16, 16)   # 256 threads per block total
blockspergrid = (ceil(N/16), ceil(N/16))
matmul_kernel[blockspergrid, threadsperblock](d_A, d_B, d_C)
```

##### **Why memory transfer matters less**

Unlike vector addition, each thread now does `N` multiplications and additions (one full dot product). With $1024\times 1024$ matrices, that is around 1024 FLOPs per thread, compared to 1 FLOP for the vector addition kernel. The compute-to-transfer ratio is much more favourable, which is why only around $17\%$ of runtime is transfer (versus $99\%$ for vector addition). This is the concept of arithmetic intensity so how much compute we get per byte transferred.

<div class="alert alert-block alert-info">

### **Week 10**

CuPy and GPU profiling

</div>


##### **GPU reduction script**

(exercise 1 of week 10)

Explaining the script:
```python
from numba import cuda

TPB = 128  # Threads per block

@cuda.jit
def reduce_kernel(data, out, n):
    # Get the 1D grid and block indices
    tid = cuda.threadIdx.x
    i = cuda.grid(1)

    # Do reduction for threadblock
    s = 1
    while s < cuda.blockDim.x:
        if tid % (2 * s) == 0 and i + s < n:
            data[i] += data[i + s]
        s *= 2
        cuda.syncthreads()  # Ensure block is synchronized

    # Write result for this block to global memory
    if tid == 0:
        out[cuda.blockIdx.x] = data[i]

def get_grid(n, tpb):
    return (n + (tpb - 1)) // tpb  # Blocks per grid

def reduce(x):
    n = len(x)
    bpg = get_grid(n, TPB)
    out = cuda.device_array(bpg, dtype=x.dtype)
    while bpg > 1:
        reduce_kernel[bpg, TPB](x, out, n)
        n = bpg
        bpg = get_grid(n, TPB)
        x[:n] = out[:n]
    reduce_kernel[bpg, TPB](x, out, n)
    return out
```

##### **Threads per block**

```python
TPB = 128
```

It chooses threads that live in one block. All threads in a thread block are placed on the same Streaming Multiprocessor (SM) and can share memory synchronization primitives.

##### **The kernel function**


```python
@cuda.jit
def reduce_kernel(data, out, n):
```

`cuda.jit` is the kernel function and `@` is used so numba compiles this to CUDA code. This is the kernel function aka the GPU entry-point that the CPU calls, and the kernel cannot return values as it runs as a GPU function, so results are written to `out`array passed by the caller. This is a fundamental constraint of the CUDA programming model described in chapter 9: *"Our function cannot return values as it is going to be implemented as a GPU kernel function, so we need to pass parameters to accommodate return values."*


##### **Thread and block identity**


```python
tid = cuda.threadIdx.x   # Local thread index within this block (0 to TPB-1)
i   = cuda.grid(1)       # Global thread index across the entire grid
```

1. Each thread figures out *who it is*.
2. `tid` is the thread's position inside its block (used to control the reduction logic).
3. `i` is the thread's position in the full array (used to access the right data element).

`cuda.grid(1)` is shorthand for `cuda.blockIdx.x * cuda.blockDim.x + cuda.threadIdx.x`. Every thread runs the same code but on a different element - this is the SIMT (Single Instruction, Multiple Thread) model that makes GPUs fast.


##### **Reduction loop in global memory**

This is the reduction loop in global memory.

```python
s = 1
while s < cuda.blockDim.x:
    if tid % (2 * s) == 0 and i + s < n:
        data[i] += data[i + s]
    s *= 2
    cuda.syncthreads()
```

1. Thread 0 adds element 0 + element 1. Thread 2 adds element 2 + element 3$ and so on.
2. In the next round the thread 0 adds element 0 + element 2. Thread 4 adds element 4 + element 6 and so on.
3. Continue halving until only thread 0 in each block holds the block's sum.

So this is the tree-reduction pattern explained below:

\- Round 1 $(s=1): n/2$ threads each do one addition.

\- Round 2 $(s=2): n/4$ threads each do one addition.

\- Then so on and so on until one value survives per block.

`cuda.syncthreads()` is critical here, because all threads in a block are running simultaneously, so we must wait for everyone to finish their addition before starting the next round, otherwise a thread might read a value that another thread has not written yet. This is the GPU equivalent of a barrier synchronization. So as chapter 9 notes, threads in the same block can share synchronization primitives precisely because they live on the same SM.

> Note on this implementation: the reduction is done directly in global GPU memory (`data`), which is the slowest memory tier. Here, a more optimised version (hinted at in chapter 9) would first load the block's elements into shared memory (the fast SM-local L1 cache), reduce there, then write one result back to global memory.

> That classic pattern is:

> 1. Load block into shared memory.

> 2. Do reduction in shared memory.

> 3. Write one result back to global.

##### **Block result to global memory**

Writing the block result to global memory.

```python
if tid == 0:
    out[cuda.blockIdx.x] = data[i]
```

1. Only thread 0 of each block writes the final result for that block.
2. The result goes into `out[block_index]`, one entry per block.

After the reduction loop, `data[i]` (where `i` is thread 0's global position, i.e. the start of this block) holds the sum of the entire block. We write that to `out` so it can be used in the next pass.


##### **Grid calculation**

Grid calculation.

```python
def get_grid(n, tpb):
    return (n + (tpb - 1)) // tpb
```

1. Calculate how many blocks are needed to cover `n` elements with `tpb` threads each.

This is the standard ceiling division trick. As chapter 9 points out, we often cannot get `blocks * threads` to exactly equal our array size, so we round up and handle the boundary (`i + s < n`) inside the kernel.


The iterative reduction loop (host side):

```python
def reduce(x):
    n = len(x)
    bpg = get_grid(n, TPB)
    out = cuda.device_array(bpg, dtype=x.dtype)
    while bpg > 1:
        reduce_kernel[bpg, TPB](x, out, n)
        n = bpg
        bpg = get_grid(n, TPB)
        x[:n] = out[:n]
    reduce_kernel[bpg, TPB](x, out, n)
    return out
```

1. Launch the kernel: each block reduces its 128 elements to 1 value so `out` has `bpg` partial sums.
2. Copy partial sums back into `x` and repeat - now the problem is `bpg` times smaller.
3. Keep looping until only 1 block remains, then do a final pass.

This is necessary because a single kernel launch can only reduce *within* blocks - there is no built-in way to synchronize *across* blocks in CUDA. So we let the CPU orchestrate multiple passes, each one shrinking the problem. This is a direct consequence of the GPU memory hierarchy: blocks on different SMs cannot share state, so the CPU must step in as the coordinator between passes.

The `[bpg, TPB]` launch syntax is the Numba way of specifying the grid configuration: `kernel[blocks_per_grid, threads_per_block](args)`, exactly as described in chapter 9.

##### **Summary**

Summary flowchart of the reduction process:

```
Large array (size n)
       ↓
  Launch kernel: bpg blocks × 128 threads
       ↓
  Each block tree-reduces its 128 elements → 1 partial sum
       ↓
  out[] now has bpg partial sums
       ↓
  Copy out → x, repeat with smaller n
       ↓  (loop until bpg == 1)
  Final kernel: 1 block reduces last partial sums
       ↓
  out[0] = total sum
```

The key week 10 concepts at play here are: the kernel function as GPU entry point, the thread/block/grid hierarchy, synchronization within a block via `syncthreads`, the cost of global versus shared memory, and the need to design algorithms around the hardware's memory topology.

##### **CuPy script**

(exercise 2 of week 10)

Explaining the script:
```python
import cupy as cp

def distance_matrix_oneloop(p1, p2):
    p1 = cp.radians(p1)
    p2 = cp.radians(p2)

    D = cp.empty((len(p1), len(p2)))
    for i in range(len(p1)):
        dsin2 = cp.sin(0.5 * (p1[i] - p2)) ** 2
        cosprod = cp.cos(p1[i, 0]) * cp.cos(p2[:, 0])
        a = dsin2[:, 0] + cosprod * dsin2[:, 1]
        row = 2 * cp.arctan2(cp.sqrt(a), cp.sqrt(1 - a))
        D[i, :] = row

    D *= 6371  # Earth radius in km
    return D

def distance_matrix_noloop(p1, p2):
    p1 = cp.radians(p1)
    p2 = cp.radians(p2)
    dsin2 = cp.sin(0.5 * (p1[:, None, :] - p2[None, :, :])) ** 2
    cosprod = cp.cos(p1[:, None, 0]) * cp.cos(p2[None, :, 0])
    D = 2 * cp.arcsin(cp.sqrt(dsin2[:, :, 0] + cosprod * dsin2[:, :, 1]))
    D *= 6371  # Earth radius in km
    return D
```


##### **What the code computes**

Both functions compute the Haversine distance between every pair of points in `p1` and `p2`, so the great-circle distance between like GPS coordinates on a sphere. The output is a matrix `D` where `D[i, j]` is the distance in kilometers between point `i` in `p1` and point `j` and `p2`.

##### **Explaining `distance_matrix_oneloop`**

It iterates over rows of `p1` in a Python loop. Each iteration here uses now CuPy vectorised operations across all of `p2`. This is a classic "one loop left" pattern from the Numpy broadcasting from week 4.

##### **Explaining distance_matrix_noloop** 

This one is fully vectorised using broadcasting. The key trick is `p1[:, None, :]` and `p2[None, :, :]`, which expand the arrays to shape `(n1, 1, 2)` and `(1, n2, 2)` respectively, causing NumPy and CuPy to broadcast the subtraction across all pairs at once, no Python loop at all.

##### **Converting to CuPy**

The conversion from NumPy to CuPy is intentionally near-trivial. We just replaced `import numpy as np` with `import cupy as cp`, and swap `np.` calls for `cp`. CuPy is a drop-in replacement for NumPy  that runs operations on the GPU instead of the CPU.

There is one critical consideration which is where the data lives. As chapter 9 explains, the CPU (host) and GPU (device) have separate memory banks. Here, a CuPy array lives in GPU memory and a NumPy array lives in CPU memory. We convert between them with:

```python
x_gpu = cp.asarray(x_cpu)   # Host to Device (HtoD)
x_cpu = x_gpu.get()         # Device to Host (DtoH)
```

This transfer happens over the PCIe bus and is expensive. Chapter 9 notes that it can take more time than the computation itself, especially for smaller workloads.

##### **Why a one-loop version will be slow on GPU**

This is the most important insight from the exercise of week 10. The `oneloop` version does this:

```python
for i in range(len(p1)):       # Python loop - CPU
    row = ...cp operations...  # GPU computation
    D[i, :] = row              # write back
```

Every iteration of that loop involves:

1. A kernel launch on the GPU for the CuPy operations.
2. Potentially a synchronisation / memory interaction.

With 5000 rows, that is 5000 separate GPU kernel launches, each doing a relatively small amount of work. The overhead of launching kernels and the Python loop itself dominates. When we profile with `nsys`, we will see a lot of `cuMemcpyHtoD` and `cuMemcpyDtoH` calls - the transfer cost dwarfing the actual compute time, exactly as the lecture slides illustrate.

This directly reflects the warning from chapter 9 stating: *"The cost of transferring data to and from the GPU memory can have a massive effect on performance, especially if the amount of computation that we do on the GPU is limited."*


##### **Why the no-loop version will be much faster**

The `noloop` version does everything in one CuPy call. The entire `(5000, 5000, 2)` broadcasting computation is dispatched to the GPU as a single kernel. This means:

> One HtoD transfer of the input data.

> One large parallel GPU computation.

> One DtoH transfer of the result matrix.

The GPU thrives here because it has thousands of cores that can process the array elements simultaneously. `nsys` will show far fewer memory operations and much more time spent in actual kernel execution. The GPU's throughput advantage is fully exploited when given one big problem rather than 5000 small ones.

##### **Summary of key concepts**

This exercise is a microcosm of the core GPU programming lesson from Week 10 that GPUs are not automatically fast just because we use CuPy. The bottleneck shifts depending on how we structure the work:

> One-loop version: bottleneck is the Python loop overhead + repeated small kernel launches + memory traffic. May be *slower* than the 
CPU version.

> No-loop version: one big vectorised operation → GPU throughput is actually utilised. Should be significantly faster than CPU for 5000×5000.

The `nsys` profiler is the tool to confirm this. Sections to look at are the CUDA API Summary (`cuMemcpyHtoD`, `cuMemcpyDtoH`, `cuLaunchKernel` call counts and times) and the GPU MemOps Summary - the ratio of time spent in memory transfers vs. actual computation tells the whole story.

<div class="alert alert-block alert-info">

### **Week 11**

HPC Workflows: Job Arrays and Job Dependencies
</div>

##### **Job arrays**



(exercise 1 of week 11)

A job array submits many identical jobs in one `bsub` call, each differing only by an index. The key directive is `#BSUB -J name[1-182]` which creates 182 jobs with indices 1 through 182. Each job gets the same resources and runs the same script, but the environment variable `$LSB_JOBINDEX` holds the unique index so the script knows which piece of work to process.

```bash
#!/bin/bash
#BSUB -J subhist[1-203]
#BSUB -q hpc
#BSUB -W 15
#BSUB -n 1
#BSUB -R "span[hosts=1]"
#BSUB -R "rusage[mem=512MB]"
#BSUB -o batch_output/subhist_%J_%I.out
#BSUB -e batch_output/subhist_%J_%I.err

source /dtu/projects/02613_2025/conda/conda_init.sh
conda activate 02613

python huedir.py $LSB_JOBINDEX
```

The `%I` placeholder in the output filename expands to the job array index, giving each job its own distinct log file. Without `%I`, all 203 jobs would write to the same file and interleave output, making debugging impossible. `%J` is the job ID shared across all array elements.

Three index syntaxes are supported:
```bash
#BSUB -J array[1-5]       # Contiguous range
#BSUB -J array[1-5:2]     # Every 2nd: 1, 3, 5
#BSUB -J array[2,29,71]   # Explicit list
```

**Email warning**: do not add `#BSUB -N` notifications to job arrays. A 200-job array sends 400 emails on start and finish, which the DTU mail server rejects if more than 100 arrive in a short interval.


##### **Monitoring job arrays**


```bash
bjobs -A                # Compact summary: PEND/RUN/DONE/EXIT counts per array
bpeek 702576[3]         # Live output of array element 3
bkill 702576[3]         # Kill one element
bkill 702576            # Kill entire array
```

`bjobs -A` is the right tool for arrays because the standard `bstat` shows one row per active element, which floods the screen for large arrays. The `DONE`/`EXIT` columns in `bjobs -A` are how we see how many elements have succeeded or crashed at a glance.


##### **Job dependencies**

(exercise 1 of week 11, continued)

The `-w` directive makes a job pend until another job reaches a specified state. Four conditions:

```bash
#BSUB -w jobname          # Wait for job named "jobname" to reach DONE
#BSUB -w done(jobname)    # Same as above, explicit
#BSUB -w exit(jobname)    # Wait for EXIT (crashed or killed)
#BSUB -w ended(jobname)   # Wait for DONE or EXIT (finished for any reason)
```

You can also depend on a job ID instead of a name: `#BSUB -w done(1234567)`.

The key distinction for the exercises: `done()` requires successful completion only, while `ended()` fires on both success and failure. Use `done()` when you only want to run the next stage if everything worked. Use `ended()` if you want the next job to run regardless (e.g., a cleanup job).

##### **Job dependencies with arrays**


When both the upstream and downstream jobs are arrays, we often want element-level dependencies so element 3 of `array2` waits only for element 3 of `array1` to finish, not all 10. The syntax uses `[*]` as a wildcard that matches corresponding indices:

```bash
#!/bin/bash
#BSUB -J array2[1-5]
#BSUB -q hpc
#BSUB -W 5
#BSUB -n 1
#BSUB -R "span[hosts=1]"
#BSUB -R "rusage[mem=512MB]"
#BSUB -w "done(array1[*])"    # Each element waits for its corresponding element
#BSUB -o jobname_%J_%I.out
#BSUB -e jobname_%J_%I.err
```

This requires both arrays to have the same length. If `array1` has 10 elements and `array2` has 5, the mapping breaks down and the dependency will not work as expected.

To wait for an entire array (not element-by-element), just reference the array name without `[*]`:

```bash
#BSUB -w done(subhist)    # Wait for ALL elements of the "subhist" array
```

This is the pattern for the face colors exercise: the histogram plotting job waits for every folder-processing job in the array to succeed before aggregating.


##### **Monitoring dependencies**


```bash
bjdepinfo 456           # Show what job 456 depends on (parents)
bjdepinfo -c 456        # Show what depends on job 456 (children)
bjdepinfo 456[2]        # Dependencies for array element 2
```

This is the debugging tool when a dependent job refuses to start. It shows the parent's current state so we can see if the dependency condition has been met yet.

##### **Face colors exercise**


(exercise 2 of week 11)

This exercise puts job arrays and dependencies together into a real pipeline. The computation, building a hue histogram over 200,000 celebrity images, is a classic map-reduce pattern:

**Map**: process each folder independently and save a partial result.
**Reduce**: load all partial results and aggregate into a final histogram.

The map step is embarrassingly parallel so no communication between jobs. Each array element processes one folder and saves `subhist_<i>.npy`. The reduce step requires all partial results to exist before it starts, enforced by the `done(subhist)` dependency.

**Python program for the map step (`huedir.py`)**:

```python
import os
from os.path import join
import sys
import numpy as np
from PIL import Image

def huehist(image):
    bins = np.linspace(0, 255, 64 + 1)
    hsv_image = np.asarray(Image.fromarray(image).convert('HSV'))
    hue_values = hsv_image[:, :, 0].reshape(-1)
    hue_hist = np.histogram(hue_values, bins)[0]
    return hue_hist

if __name__ == '__main__':
    idx = int(sys.argv[1]) - 1  # Job arrays are 1-indexed; folders are 0-indexed
    base_path = '/dtu/projects/02613_2024/data/celeba/images'
    folders = sorted(os.listdir(base_path))
    images = sorted(os.listdir(join(base_path, folders[idx])))

    hist = 0
    for im in images:
        image = Image.open(join(base_path, folders[idx], im))
        image = np.array(image)
        hist += huehist(image)

    np.save(f"subhist_{idx}.npy", hist)
```

The index offset `int(sys.argv[1]) - 1` is a recurring pattern. `$LSB_JOBINDEX` starts at 1 (LSF convention), but Python list indices start at 0. Forgetting this off-by-one means job 1 processes folder 0, job 2 processes folder 1, and the last folder is never processed.

Accumulating the per-image histograms with `hist += huehist(image)` works because histograms are additive. Summing them is a valid reduction because histogram binning is a linear operation.

**Python program for the reduce step (`plothuehist.py`)**:

```python
from glob import glob
import numpy as np
import matplotlib.pyplot as plt

if __name__ == '__main__':
    histfiles = glob('subhist_*.npy')
    hist = 0
    for hf in histfiles:
        hist += np.load(hf)
    plt.bar(np.linspace(0, 255, 64), hist, width=4)
    plt.savefig('histogram.png')
```

**Job script for the reduce step (`plothist.sh`)**:

```bash
#!/bin/bash
#BSUB -J plothist
#BSUB -q hpc
#BSUB -W 5
#BSUB -n 1
#BSUB -w done(subhist)
#BSUB -R "span[hosts=1]"
#BSUB -R "rusage[mem=512MB]"
#BSUB -o batch_output/plothist_%J.out
#BSUB -e batch_output/plothist_%J.err

source /dtu/projects/02613_2024/conda/conda_init.sh
conda activate 02613

python plothuehist.py
```

The result shows that red and orange hues dominate the dataset, which makes intuitive sense as human skin tones occupy that region of the hue spectrum.

The pipeline structure illustrated here generalises to virtually any large-scale data processing task: split the data into independent chunks, submit a job array to process them, submit a dependent job to aggregate. This is the standard HPC workflow pattern from the week 11 lecture.
```

<div class="alert alert-block alert-info">

### **Week 12**

Numba CPU (exercise context)
</div>


#### **Exercise: Numba on the matmul loop (week 9 carry-over)**



The week 9 / week 12 exercise adds `@jit(nopython=True)` to the triple-loop matrix multiplication from week 9:

```python
from numba import jit

@jit(nopython=True)
def matmul_jit(A, B, C):
    for i in range(A.shape[0]):
        for k in range(A.shape[1]):
            for j in range(B.shape[1]):
                C[i, j] += A[i, k] * B[k, j]
```

**Why this is faster than the plain Python version:** every iteration of the inner loop in pure Python involves interpreter overhead - type checking, object lookup, reference counting. `@jit(nopython=True)` compiles the entire triple loop to native machine code, eliminating all of that. The observed speedup is around 470× over pure Python.

**Why it is still slower than `np.dot`:** `np.dot` calls a highly optimised BLAS `DGEMM` routine that uses SIMD vector instructions, multi-level cache tiling, and decades of hardware-specific tuning. Numba JIT produces generic machine code - it cannot automatically apply those tricks. Numba wins when you have custom loop logic that cannot be expressed as a NumPy operation.

**The warm-up rule in practice:**
```python
# Always do this pattern when benchmarking Numba:
C = np.zeros((N, N))
matmul_jit(A, B, C)   # warm-up call - triggers compilation, do NOT time this

C = np.zeros((N, N))
t0 = perf_counter()
matmul_jit(A, B, C)   # this is the real timing
t1 = perf_counter()
```


#### **The parallel Mandelbrot / Numba `prange` exercise**


The Mandelbrot from week 5 can be made parallel with `@njit(parallel=True)` and `prange` on the outer loop:

```python
from numba import njit, prange

@njit(parallel=True)
def mandelbrot_parallel(size, max_iter, img):
    for xp in prange(size):            # parallelised - each column is independent
        x = ...
        for yp in range(size):         # NOT parallelised - inner loop is fine serial
            y = ...
            img[yp, xp] = escape(x, y, max_iter)
```

The outer loop over `xp` is safe to parallelise because each column `img[:, xp]` is written by exactly one thread. The inner loop over `yp` is serial because all writes go to the same column. This gives ~3× speedup on 8 cores.



#### **Why a function using a Python dict cannot be `@njit`**





The F25 exam Q22 pattern: a simulation function that takes a `params` dictionary object.

```python
def simulate_ball(params):
    while params.height > 0:          # Python attribute access on Python object
        params = simulate_one_step(params)   # Python function returning Python dict
    save_results(params)
```

This cannot be decorated with `@jit(nopython=True)` because:
1. `params` is a Python object - Numba cannot determine its type at compile time.
2. `params.height` is a Python attribute lookup - not a typed memory access.
3. The function it calls (`simulate_one_step`) is also not `@njit`.

**The fix:** extract scalar fields from the dict before entering the compiled function:
```python
@njit
def simulate_core(height, speed, mass, ...):   # only scalars/arrays
    while height > 0:
        height, speed = one_step(height, speed, mass, ...)
    return height, speed

# Caller extracts from dict, calls compiled function:
h, s = simulate_core(params['height'], params['speed'], params['mass'])
```

<div class="alert alert-block alert-info">

### **Week 13**

HPC Pitfalls: Excessive I/O and Multi-Threading
</div>


##### **Excessive I/O**

(exercise 1 of week 13)

The LSF `-o/-e` channels for capturing stdout and stderr are convenient but surprisingly slow. They route output through a special mechanism inside the batch system which incurs significant overhead for each line written. When a program produces a large volume of output (like printing 100,000 lines), the `-o/-e` channels become the bottleneck, not the computation itself.

The difference between the two job scripts is simply where the output goes:

**Standard approach (slow for heavy output)**:
```bash
#!/bin/bash
#BSUB -J print
#BSUB -q hpc
#BSUB -W 00:10
#BSUB -n 1
#BSUB -R "select[model==XeonE5_2650v4]"
#BSUB -R "rusage[mem=512MB]"
#BSUB -o /work3/02613/dump/printlots_%J.out
#BSUB -e /work3/02613/dump/printlots_%J.err

source /dtu/projects/02613_2024/conda/conda_init.sh
conda activate 02613

python -u printlots.py
```

**Manual redirection (fast)**:
```bash
#!/bin/bash
#BSUB -J print
#BSUB -q hpc
#BSUB -W 00:10
#BSUB -n 1
#BSUB -R "select[model==XeonE5_2650v4]"
#BSUB -R "rusage[mem=512MB]"
#BSUB -o /work3/02613/dump/printlots_%J.out
#BSUB -e /work3/02613/dump/printlots_%J.err

source /dtu/projects/02613_2024/conda/conda_init.sh
conda activate 02613

python -u printlots.py \
    1> /work3/02613/dump/output_${LSB_JOBID}.txt \
    2> /work3/02613/dump/error_${LSB_JOBID}.txt
```

In the second script, `1>` redirects stdout and `2>` redirects stderr directly to files on the `/work3` filesystem, bypassing the `-o/-e` channels entirely. The `-o/-e` files are kept but will just contain the LSF job summary at the end since no program output flows through them.

The `${LSB_JOBID}` variable gives the job ID inside the script, the bash equivalent of `%J` in the `#BSUB` directive. This is used to give the output file a unique name matching the job.

##### **The `-u` flag (unbuffered output)**


Python buffers `print()` output by default when writing to a file (as opposed to a terminal). This means output is held in memory and written in large batches. With `-u`, Python flushes every `print()` call immediately. Without `-u`, the output might not appear until the job finishes or the buffer fills. This is why `-u` is always used in HPC job scripts: it guarantees output is visible via `bpeek` while the job is still running, and ensures all output is flushed even if the job is killed.


##### **The runtime difference**


The resource summaries in the output files reveal the full cost:

| | Without manual redirect | With manual redirect |
|---|---|---|
| Run time | 80 s | 3 s |
| CPU time | 7.37 s | 1.48 s |
| Max Memory | 6 MB | - |

This is a **26x speedup** from changing just one line of the job script. The `Run time` (wall clock) difference is so dramatic because the `-o/-e` channel involves the LSF daemon writing to a shared distributed filesystem with high per-write latency. With 100,000 lines of output, that latency adds up catastrophically. The CPU time is also lower because less time is spent waiting on I/O system calls.

The caveat from the lecture is important: this effect is only visible on `/work3`. The home directories use a different filesystem where the difference may not appear. Always use `/work3/02613/dump/` for output-heavy experiments.

##### **NumPy multi-threading**


(exercise 2 of week 13)

This exercise exposes a common HPC pitfall: NumPy performs linear algebra operations (matrix multiply, dot products, SVD, etc.) through a backend called BLAS (Basic Linear Algebra Subprograms). The BLAS library is multithreaded and releases Python's GIL, meaning it can use real CPU parallelism. However, this threading is not controlled via Python at all - it is controlled through environment variables set before the Python process starts.


##### **Why requesting more cores does not help by itself**


Submitting with 1 core versus 8 cores makes no difference to runtime ($5.87 \: s$ versus $5.96 \: s$). Simply requesting more cores from LSF makes them available to the job, but does not tell NumPy's BLAS backend to actually use them. BLAS defaults to 1 thread unless explicitly told otherwise.

##### **Enabling multi-threading via environment variables**


Since the exact threading library depends on which BLAS implementation NumPy is linked against (OpenBLAS, MKL, ATLAS, etc.), the safest approach is to set all relevant variables at once in the job script:

```bash
#!/bin/bash
#BSUB -J matmuls
#BSUB -q hpc
#BSUB -W 00:10
#BSUB -n 8
#BSUB -R "span[hosts=1]"
#BSUB -R "select[model==XeonE5_2650v4]"
#BSUB -R "rusage[mem=4GB]"
#BSUB -o batch_output/matmuls_%J.out
#BSUB -e batch_output/matmuls_%J.err

source /dtu/projects/02613_2024/conda/conda_init.sh
conda activate 02613

NUM_THREADS=8
OMP_NUM_THREADS=$NUM_THREADS
MPI_NUM_THREADS=$NUM_THREADS
MKL_NUM_THREADS=$NUM_THREADS
OPENBLAS_NUM_THREADS=$NUM_THREADS

python -u matmuls.py
```

This drops runtime from $5.87 \: s$ to $1.28 \: s$, a 4.6x speedup with 8 threads. Not perfectly linear due to communication overhead and Amdahl's law on the serial parts of the computation, but still a substantial win. To disable threading later, just set `NUM_THREADS=1` or remove the variable exports.

The reason `np.matmul` is parallelisable this way is that matrix multiplication is a BLAS Level 3 routine (`DGEMM`). These operations have high arithmetic intensity (many FLOPs per byte transferred), making them ideal candidates for multi-core throughput. Simple element-wise NumPy operations would see little benefit.

##### **ThreadPool parallelism alongside BLAS threading**


Using a `ThreadPool` to parallelise the outer loop over 100 matrix multiplications works because NumPy releases the GIL during BLAS calls. Since each thread spends almost all its time inside `np.matmul` (with GIL released), multiple threads can run truly simultaneously, unlike pure Python loops which would serialise on the GIL.

```python
from multiprocessing.pool import ThreadPool

def matmuls(A, B):
    n = A.shape[0]
    with ThreadPool(8) as p:
        C = np.concatenate(p.starmap(np.matmul, zip(A, B)))
    return C
```

`p.starmap(np.matmul, zip(A, B))` maps the function across pairs `(A[0], B[0])`, `(A[1], B[1])`, ..., equivalent to the original for-loop but dispatched across 8 threads. `np.concatenate` re-assembles the list of result matrices back into the `(100, 1000, 1000)` output array.

This is why `ThreadPool` is valid here but would not be for pure Python computation: threads share memory (no pickling overhead, unlike multiprocessing) and the GIL is not a bottleneck because NumPy releases it for the heavy computation.

##### **The thread stacking problem**



When both `ThreadPool(8)` and BLAS threading with 8 threads are active simultaneously, we get $8 \times 8 = 64$ concurrent threads on 8 cores. Each thread in the pool spawns 8 BLAS threads, all competing for the same CPU cores. The OS scheduler has to constantly switch between all 64 threads, wasting time on context switches rather than computation. Runtime degrades to $1.87 \: s$ - slower than either approach alone.

The fix is to disable BLAS threading when using a ThreadPool (set `NUM_THREADS=1`), letting the ThreadPool provide all the parallelism:

| Configuration | Runtime |
|---|---|
| 1 core, no BLAS threading | $5.87 \: s$ |
| 8 cores, no BLAS threading | $5.96 \: s$ (no change) |
| 8 cores, BLAS threading × 8 | $1.28 \: s$ |
| 8 cores, ThreadPool(8) + BLAS × 8 | $1.87 \: s$ (too many threads) |
| 8 cores, ThreadPool(8) + BLAS × 1 | $1.33 \: s$ |

The lesson generalises well beyond this exercise: whenever writing parallel code, check what your dependencies are doing with threads. A library that "just works" in serial may silently spin up its own thread pool, causing unintended thread oversubscription when you also parallelise at the outer level. The rule of thumb is that the total number of threads running at any moment should not exceed the number of physical cores.